# 📓 Notebook 2 — Chunking Strategies & Embeddings

**Clase: RAG - Retrieval Augmented Generation**  
**Duración estimada:** 40 minutos  

---

## 🎯 Objetivos

- Entender por qué el chunking es uno de los factores más críticos en un pipeline RAG
- Comparar 4 estrategias de chunking distintas
- Comparar embeddings densos (OpenAI vs HuggingFace local)
- Visualizar el impacto de cada estrategia sobre los documentos recuperados

---

## 🧠 Teoría: ¿Por qué importa el chunking? (10 min)

Cuando indexamos documentos en un vector store, no metemos el documento entero: lo **troceamos** en fragmentos (chunks) y embeddemos cada uno por separado.

El problema: **si el chunk es demasiado grande**, pierde precisión semántica en el embedding y contamina el contexto del LLM. **Si es demasiado pequeño**, puede perder contexto necesario para responder.

```
Documento original
        │
        ▼
  ┌─────────────┐
  │  Splitter   │  ← aquí está el 80% de la magia
  └─────────────┘
        │
   [chunk1] [chunk2] [chunk3] ...
        │
   Embedding model
        │
   Vector store (Chroma)
        │
   Retrieval → LLM → Respuesta
```

### Parámetros clave

| Parámetro | Descripción | Valor típico |
|-----------|-------------|---------------|
| `chunk_size` | Tamaño máximo del chunk (en caracteres o tokens) | 500–1500 |
| `chunk_overlap` | Solapamiento entre chunks consecutivos | 10–20% del chunk_size |

> **Regla de oro:** el overlap evita que información importante quede partida entre dos chunks.

## ⚙️ Setup e instalación

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
%pip install -q langchain langchain-community langchain-openai langchain-experimental \
               chromadb sentence-transformers pypdf tiktoken matplotlib pandas

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Configura tu API key de OpenAI
# os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- descomenta y pon tu key

# Verificar que está configurada
assert os.environ.get("OPENAI_API_KEY"), "⚠️ Configura OPENAI_API_KEY antes de continuar"
print("✅ API key configurada")

## 📄 1. Cargar el documento de prueba

Usaremos un PDF largo para que las diferencias entre estrategias sean evidentes.  
Puedes usar cualquier PDF propio o descargar uno de ejemplo.

In [ ]:
# Opción A: cargar un PDF local
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("tu_documento.pdf")

# Opción B: descargar el paper original de RAG de Meta (usado en clase)
import urllib.request
url = "https://arxiv.org/pdf/2005.11401"
urllib.request.urlretrieve(url, "rag_paper.pdf")
print("✅ PDF descargado: rag_paper.pdf")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("rag_paper.pdf")
pages = loader.load()

print(f"📄 Páginas cargadas: {len(pages)}")
print(f"📝 Caracteres en página 1: {len(pages[0].page_content)}")
print(f"\n--- Muestra de página 1 ---\n{pages[0].page_content[:500]}")

In [ ]:
# Unir todas las páginas en un solo texto para algunos splitters
from langchain.schema import Document

full_text = "\n\n".join([p.page_content for p in pages])
full_doc = [Document(page_content=full_text, metadata={"source": "rag_paper.pdf"})]

print(f"📊 Total caracteres en el documento: {len(full_text):,}")
print(f"📊 Total palabras aproximadas: {len(full_text.split()):,}")

---

## ✂️ 2. Las 4 estrategias de chunking

### Estrategia 1: `CharacterTextSplitter` — el más simple

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

# Divide por un separador fijo (por defecto '\n\n')
splitter_char = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)

chunks_char = splitter_char.split_documents(full_doc)

print(f"🔢 Chunks generados: {len(chunks_char)}")
print(f"📏 Tamaño medio: {sum(len(c.page_content) for c in chunks_char) / len(chunks_char):.0f} caracteres")
print(f"\n--- Chunk de ejemplo ---\n{chunks_char[5].page_content[:400]}")

### Estrategia 2: `RecursiveCharacterTextSplitter` — el estándar recomendado

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Intenta dividir por párrafos, luego frases, luego palabras, luego caracteres
# Jerarquía de separadores: ['\n\n', '\n', ' ', '']
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # jerarquía explícita
)

chunks_recursive = splitter_recursive.split_documents(full_doc)

print(f"🔢 Chunks generados: {len(chunks_recursive)}")
print(f"📏 Tamaño medio: {sum(len(c.page_content) for c in chunks_recursive) / len(chunks_recursive):.0f} caracteres")
print(f"\n--- Chunk de ejemplo ---\n{chunks_recursive[5].page_content[:400]}")

### Estrategia 3: `SemanticChunker` — divide por significado, no por tamaño

> 💡 Este splitter usa embeddings para detectar **cambios semánticos** entre frases. Divide donde el significado cambia, no donde el texto llega a N caracteres.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# breakpoint_threshold_type puede ser: 'percentile', 'standard_deviation', 'interquartile'
splitter_semantic = SemanticChunker(
    embeddings=embeddings_openai,
    breakpoint_threshold_type="percentile",  # divide donde la distancia semántica supera el percentil 95
    breakpoint_threshold_amount=95,
)

# ⚠️ Nota: este splitter hace llamadas a la API de OpenAI para calcular embeddings
chunks_semantic = splitter_semantic.split_documents(full_doc)

print(f"🔢 Chunks generados: {len(chunks_semantic)}")
print(f"📏 Tamaño medio: {sum(len(c.page_content) for c in chunks_semantic) / len(chunks_semantic):.0f} caracteres")
print(f"\n--- Chunk de ejemplo ---\n{chunks_semantic[5].page_content[:400]}")

### Estrategia 4: `MarkdownHeaderTextSplitter` — divide respetando la estructura

> 💡 Ideal cuando el documento tiene estructura clara (Markdown, HTML, código). Preserva los headers como metadata.

In [ ]:
from langchain.text_splitter import MarkdownHeaderTextSplitter

# Ejemplo con texto Markdown estructurado
markdown_doc = """
# Introducción a RAG

RAG combina recuperación de documentos con generación de texto.
Es especialmente útil para preguntas sobre documentos específicos.

## Componentes principales

Los componentes de un sistema RAG son el retriever y el generador.
El retriever busca documentos relevantes en una base vectorial.

### El Retriever

El retriever convierte la query en un vector y busca los más cercanos.
Usa métricas como cosine similarity o producto escalar.

### El Generador

El generador (LLM) recibe el contexto recuperado y genera la respuesta.
Modelos como GPT-4 o Claude son opciones comunes.

## Evaluación

Evaluar RAG requiere métricas especializadas como RAGAS.
Las métricas principales son faithfulness y answer relevancy.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter_md = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
chunks_md = splitter_md.split_text(markdown_doc)

print(f"🔢 Chunks generados: {len(chunks_md)}")
for i, chunk in enumerate(chunks_md):
    print(f"\n--- Chunk {i+1} ---")
    print(f"📌 Metadata: {chunk.metadata}")
    print(f"📝 Contenido: {chunk.page_content[:200]}")

---

## 📊 3. Comparativa visual de los chunkers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Recopilar métricas
strategies = {
    "CharacterTextSplitter": chunks_char,
    "RecursiveCharacter\nTextSplitter": chunks_recursive,
    "SemanticChunker": chunks_semantic,
}

stats = []
for name, chunks in strategies.items():
    sizes = [len(c.page_content) for c in chunks]
    stats.append({
        "Estrategia": name,
        "Nº chunks": len(chunks),
        "Tamaño medio": np.mean(sizes),
        "Tamaño mínimo": np.min(sizes),
        "Tamaño máximo": np.max(sizes),
        "Desviación estándar": np.std(sizes),
    })

df_stats = pd.DataFrame(stats).set_index("Estrategia")
print(df_stats.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, (name, chunks), color in zip(axes, strategies.items(), colors):
    sizes = [len(c.page_content) for c in chunks]
    ax.hist(sizes, bins=30, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(np.mean(sizes), color="red", linestyle="--", linewidth=1.5, label=f"Media: {np.mean(sizes):.0f}")
    ax.set_title(name.replace("\n", " "), fontsize=11, fontweight="bold")
    ax.set_xlabel("Tamaño del chunk (caracteres)")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f"n={len(chunks)}", transform=ax.transAxes,
            ha="right", va="top", fontsize=10, color="gray")

plt.suptitle("Distribución del tamaño de chunks por estrategia", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("chunk_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Gráfica guardada como chunk_comparison.png")

---

## 🤗 4. Comparativa de modelos de embeddings

Vamos a comparar dos modelos de embeddings:
- **`text-embedding-3-small`** (OpenAI) — de pago, 1536 dimensiones, muy bueno
- **`all-MiniLM-L6-v2`** (HuggingFace) — gratuito, 384 dimensiones, rápido y local

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Modelo OpenAI
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# Modelo HuggingFace local (se descarga la primera vez ~90MB)
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("✅ Modelos de embeddings cargados")

In [ ]:
import time
import numpy as np

# Frases de prueba para comparar semántica
test_sentences = [
    "RAG combines retrieval with generation",
    "Retrieval Augmented Generation uses document search",
    "The Eiffel Tower is in Paris",
    "Machine learning models need training data",
    "Neural networks learn from examples",
]

# Embeddings con OpenAI
t0 = time.time()
emb_openai = embeddings_openai.embed_documents(test_sentences)
time_openai = time.time() - t0

# Embeddings con HuggingFace
t0 = time.time()
emb_hf = embeddings_hf.embed_documents(test_sentences)
time_hf = time.time() - t0

print(f"📐 Dimensiones OpenAI:      {len(emb_openai[0])}")
print(f"📐 Dimensiones HuggingFace: {len(emb_hf[0])}")
print(f"\n⏱️  Tiempo OpenAI:      {time_openai:.2f}s")
print(f"⏱️  Tiempo HuggingFace: {time_hf:.2f}s")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def plot_similarity_matrix(embeddings, labels, title, ax, cmap="Blues"):
    """Dibuja la matriz de similitud coseno entre embeddings."""
    sim_matrix = cosine_similarity(embeddings)
    im = ax.imshow(sim_matrix, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    short_labels = [l[:25] + "..." if len(l) > 25 else l for l in labels]
    ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(short_labels, fontsize=8)
    ax.set_title(title, fontweight="bold", pad=10)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center",
                   fontsize=8, color="black" if sim_matrix[i,j] < 0.7 else "white")
    return im

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plot_similarity_matrix(emb_openai, test_sentences, "OpenAI text-embedding-3-small", ax1, cmap="Blues")
im = plot_similarity_matrix(emb_hf, test_sentences, "HuggingFace all-MiniLM-L6-v2", ax2, cmap="Greens")

plt.suptitle("Matrices de similitud coseno entre frases de prueba", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("embedding_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n💡 Cuanto más azul/verde, más similares semánticamente son las frases.")
print("   Las frases 0 y 1 sobre RAG deberían tener alta similitud.")
print("   La frase sobre la Torre Eiffel debería ser muy diferente.")

---

## 🔍 5. Impacto del chunking en el retrieval

Ahora indexamos los chunks de cada estrategia en **Chroma separadas** y comparamos qué recupera cada una ante la misma query.

In [ ]:
from langchain_community.vectorstores import Chroma
import shutil

def build_vectorstore(chunks, embeddings, persist_dir):
    """Crea un Chroma vectorstore a partir de una lista de chunks."""
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )

# Usamos HuggingFace para no gastar créditos en el indexado
print("Indexando CharacterTextSplitter...")
db_char = build_vectorstore(chunks_char, embeddings_hf, "./chroma_char")

print("Indexando RecursiveCharacterTextSplitter...")
db_recursive = build_vectorstore(chunks_recursive, embeddings_hf, "./chroma_recursive")

print("Indexando SemanticChunker...")
db_semantic = build_vectorstore(chunks_semantic, embeddings_hf, "./chroma_semantic")

print("\n✅ Los 3 vectorstores están listos")

In [ ]:
def compare_retrieval(query, vectorstores_dict, k=3):
    """Compara los chunks recuperados por cada estrategia ante una misma query."""
    print(f"\n{'='*70}")
    print(f"🔍 QUERY: {query}")
    print(f"{'='*70}")

    for name, db in vectorstores_dict.items():
        docs = db.similarity_search(query, k=k)
        print(f"\n📚 [{name}] — {len(docs)} chunks recuperados")
        print("-" * 50)
        for i, doc in enumerate(docs):
            print(f"  Chunk {i+1} ({len(doc.page_content)} chars): {doc.page_content[:180].strip()}...")

vectorstores = {
    "CharacterTextSplitter": db_char,
    "RecursiveCharacterTextSplitter": db_recursive,
    "SemanticChunker": db_semantic,
}

# Probar con distintas queries
queries = [
    "How does RAG combine retrieval with generation?",
    "What are the limitations of standard language models?",
    "How is the retriever trained in RAG?",
]

for query in queries:
    compare_retrieval(query, vectorstores, k=2)

---

## ✏️ EJERCICIO (15 min)

### Objetivo: Determinar qué estrategia de chunking recupera mejor información

**Instrucciones:**

1. Elige un documento de tu elección (PDF o texto largo) o usa el que ya tenemos
2. Crea vectorstores con **3 configuraciones distintas** de `RecursiveCharacterTextSplitter` variando `chunk_size` y `chunk_overlap`
3. Formula **3 preguntas** sobre el documento que requieran información específica
4. Compara los chunks recuperados y razona cuál es mejor

**Completa el código a continuación:**

In [ ]:
# ✏️ EJERCICIO — Completa el código

# PASO 1: Define 3 configuraciones de splitter con distintos chunk_size y chunk_overlap
configs = [
    {"chunk_size": ???, "chunk_overlap": ???, "name": "Config A - Small chunks"},
    {"chunk_size": ???, "chunk_overlap": ???, "name": "Config B - Medium chunks"},
    {"chunk_size": ???, "chunk_overlap": ???, "name": "Config C - Large chunks"},
]

# PASO 2: Crea los splitters, genera chunks e indexa en Chroma
exercise_vectorstores = {}
for config in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    chunks = splitter.split_documents(full_doc)  # o tu documento
    db = build_vectorstore(chunks, embeddings_hf, f"./chroma_{config['name'].split()[2].lower()}")
    exercise_vectorstores[config["name"]] = db
    print(f"✅ {config['name']}: {len(chunks)} chunks")

# PASO 3: Define 3 preguntas sobre el documento
my_queries = [
    "???",  # pregunta 1
    "???",  # pregunta 2
    "???",  # pregunta 3
]

# PASO 4: Compara los resultados
for query in my_queries:
    compare_retrieval(query, exercise_vectorstores, k=2)

In [ ]:
# PASO 5 (reflexión): Rellena este análisis con tus observaciones
conclusions = """
📝 MIS CONCLUSIONES:

Pregunta 1:
- Mejor configuración: ???
- Motivo: ???

Pregunta 2:
- Mejor configuración: ???
- Motivo: ???

Pregunta 3:
- Mejor configuración: ???
- Motivo: ???

Conclusión general:
- Configuración recomendada para este documento: ???
- ¿Qué factor impactó más: chunk_size o chunk_overlap? ???
"""
print(conclusions)

---

## 🔑 Resumen: ¿Cuándo usar cada estrategia?

| Estrategia | Cuándo usarla | Ventaja | Inconveniente |
|------------|--------------|---------|---------------|
| `CharacterTextSplitter` | Prototipado rápido | Muy simple | Puede cortar frases a mitad |
| `RecursiveCharacterTextSplitter` | **Caso general** (recomendado por defecto) | Respeta estructura natural | Tamaño fijo, no semántico |
| `SemanticChunker` | Documentos largos y heterogéneos | Divide por significado real | Costoso (llama a la API) |
| `MarkdownHeaderTextSplitter` | Documentación técnica, wikis, código | Preserva jerarquía como metadata | Solo funciona con Markdown/HTML |

### Reglas prácticas
- **chunk_size 500–1000** para respuestas precisas a preguntas específicas
- **chunk_size 1000–2000** para resumir o preguntas que necesitan contexto amplio
- **chunk_overlap 10–20%** del chunk_size como punto de partida
- Siempre **evalúa con RAGAS** (Notebook 4) antes de decidir

---

## ➡️ Siguiente: Notebook 3 — Reranking & Query Optimization

Ahora que sabemos chunkar bien, veremos cómo mejorar la **relevancia** de lo que le pasamos al LLM con reranking y query expansion.

In [ ]:
# Limpieza de vectorstores temporales (opcional)
import shutil

dirs_to_clean = ["./chroma_char", "./chroma_recursive", "./chroma_semantic",
                 "./chroma_small", "./chroma_medium", "./chroma_large"]

for d in dirs_to_clean:
    if os.path.exists(d):
        shutil.rmtree(d)

print("🧹 Vectorstores temporales eliminados")